<a href="https://colab.research.google.com/github/akashskypatel/NeurCross/blob/main/NeurCross_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/akashskypatel/NeurCross.git
!pip install trimesh


  Cloning https://github.com/akashskypatel/NeurCross.git (to revision optimize) to /tmp/pip-req-build-5vthnpzs
  Running command git clone --filter=blob:none --quiet https://github.com/akashskypatel/NeurCross.git /tmp/pip-req-build-5vthnpzs
  Running command git checkout -b optimize --track origin/optimize
  Switched to a new branch 'optimize'
  Branch 'optimize' set up to track remote branch 'optimize' from 'origin'.
  Resolved https://github.com/akashskypatel/NeurCross.git to commit a2b39ed2de9dacf282b99bf60a0f1653f99e5df9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import kagglehub
import os

# List of Kaggle dataset identifiers
KAGGLE_DATASET_IDS = [
    "balraj98/modelnet40-princeton-3d-object-dataset",
    "mrbiid/meshy-ai-800-glb-3d-assets-categorised-and-labelled",
    "programmer3/cadcae-design-dataset",
    "timfuger/part-processing-dataset",
    "chienru/mask2-3d",
    "sc0v1n0/3d-models",
    "lukaszfuszara/thingi10k-name-and-category",
    "amytai/nutritionverse-3d",
    "mrbiid/meshy-ai-363-ply-creatures-labelled",
]

# This list will store the actual local paths where the datasets are downloaded
LOCAL_DOWNLOADED_PATHS = []

for dataset_id in KAGGLE_DATASET_IDS:
  # kagglehub.dataset_download returns the local path to the downloaded dataset
  local_path = kagglehub.dataset_download(dataset_id)
  LOCAL_DOWNLOADED_PATHS.append(local_path)

# You can now use LOCAL_DOWNLOADED_PATHS to access the files
print("Local paths of downloaded datasets:")
for p in LOCAL_DOWNLOADED_PATHS:
    print(p)

Download already complete (2045989773 bytes).
Extracting files...


100%|██████████| 1.13G/1.13G [00:09<00:00, 131MB/s]

Extracting files...


100%|██████████| 1.01G/1.01G [00:06<00:00, 178MB/s]

Extracting files...


...

100%|██████████| 1.37G/1.37G [00:52<00:00, 28.2MB/s]

Extracting files...


100%|██████████| 229M/229M [00:07<00:00, 32.8MB/s]

Extracting files...


Local paths of downloaded datasets:
/root/.cache/kagglehub/datasets/balraj98/modelnet40-princeton-3d-object-dataset/versions/1
/root/.cache/kagglehub/datasets/mrbiid/meshy-ai-800-glb-3d-assets-categorised-and-labelled/versions/1
/root/.cache/kagglehub/datasets/programmer3/cadcae-design-dataset/versions/1
/root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2
/root/.cache/kagglehub/datasets/chienru/mask2-3d/versions/1
/root/.cache/kagglehub/datasets/sc0v1n0/3d-models/versions/2
/root/.cache/kagglehub/datasets/lukaszfuszara/thingi10k-name-and-category/versions/2
/root/.cache/kagglehub/datasets/amytai/nutritionverse-3d/versions/1
/root/.cache/kagglehub/datasets/mrbiid/meshy-ai-363-ply-creatures-labelled/versions/1


In [ ]:
from huggingface_hub import snapshot_download
import os

# Use the huggingface_hub API to download the dataset repository
print("Downloading dataset using Hugging Face API...")
repo_id = "akashskypatel/NeurCross"
extract_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["directional-data.zip"]
)

# Since snapshot_download returns the folder, we still need to unzip the specific file if it's compressed
zip_file_path = os.path.join(extract_dir, "directional-data.zip")
final_extract_path = "/content/directional_data_hf"

if os.path.exists(zip_file_path):
    import zipfile
    print(f"Extracting {zip_file_path} to {final_extract_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(final_extract_path)

    # Add the extracted folder to the global path list
    if final_extract_path not in LOCAL_DOWNLOADED_PATHS:
        LOCAL_DOWNLOADED_PATHS.append(final_extract_path)
        print(f"Added {final_extract_path} to LOCAL_DOWNLOADED_PATHS")
else:
    # If the files are already loose in the repo, just add the extract_dir
    if extract_dir not in LOCAL_DOWNLOADED_PATHS:
        LOCAL_DOWNLOADED_PATHS.append(extract_dir)
        print(f"Added {extract_dir} to LOCAL_DOWNLOADED_PATHS")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting /root/.cache/huggingface/hub/datasets--akashskypatel--NeurCross/snapshots/c014c38bb0b3a41538ae1299deeb6285672cda41/directional-data.zip to /content/directional_data_hf...
Added /content/directional_data_hf to LOCAL_DOWNLOADED_PATHS


In [ ]:
allowed_extensions = {'.obj', '.off', '.stl', '.ply', '.glb'}
mesh_files = []

# Iterate through the actual local paths where datasets were downloaded
for base_path in LOCAL_DOWNLOADED_PATHS:
    print(f"\nSearching in: {base_path}")
    found_files_in_current_path = False
    for root, dirs, files in os.walk(base_path):
        for file in files:
            file_extension = os.path.splitext(file)[1].lower()
            if file_extension in allowed_extensions:
                mesh_files.append(os.path.join(root, file))
                print(f"file: {os.path.join(root, file)} added")
                found_files_in_current_path = True
    if not found_files_in_current_path:
        print(f"  No matching files found in {base_path}.")

print(f"\nFound {len(mesh_files)} mesh files in total.")

Streaming output truncated to the last 5000 lines.
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Cab3-22 - Cab3-03-1.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Cab2-05 - Cab2-02-2.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Shelf2-08 - Shelf2-03-1.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Cab2-07 - Cab2-08-1.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Shelf2-02 - Shelf2-03-1.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-dataset/versions/2/Meshes/Meshes/1B_FabInc/train/CNC/Cab2-07 - Cab2-02-2.obj added
file: /root/.cache/kagglehub/datasets/timfuger/part-processing-da

In [ ]:
import neurcross
import pathlib
import subprocess
import os
import sys

out_dir = "/home/train/neurocross"
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)

# Re-check mesh_files
allowed_extensions = {'.obj', '.off', '.stl', '.ply', '.glb'}
mesh_files = []
if 'LOCAL_DOWNLOADED_PATHS' in globals():
    for base_path in LOCAL_DOWNLOADED_PATHS:
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if os.path.splitext(file)[1].lower() in allowed_extensions:
                    mesh_files.append(os.path.join(root, file))

target_mesh = None
for f_path in mesh_files:
    if f_path.endswith('.obj') or f_path.endswith('.off'):
        target_mesh = f_path
        break

if not target_mesh and mesh_files:
    target_mesh = mesh_files[0]

if target_mesh:
    print(f"Using mesh file: {target_mesh}")
    # Parameters for stability
    n_samples = 10000

    command = [
        "python",
        "-m",
        "neurcross",
        "train-quad-mesh",
        "--data_path", target_mesh,
        "--out_dir", out_dir,
        "--n_samples", str(n_samples)
    ]

    print(f"Executing command: {' '.join(command)}")

    # Use Popen to stream output in real-time
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    print("--- Real-time Training Output ---")
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()  # Ensure output is printed immediately

    process.wait()
    if process.returncode != 0:
        print(f"\nProcess failed with return code {process.returncode}")
else:
    print("No compatible mesh files found.")

Using mesh file: /root/.cache/kagglehub/datasets/balraj98/modelnet40-princeton-3d-object-dataset/versions/1/ModelNet40/dresser/train/dresser_0073.off
Executing command: python -m neurcross train-quad-mesh --data_path /root/.cache/kagglehub/datasets/balraj98/modelnet40-princeton-3d-object-dataset/versions/1/ModelNet40/dresser/train/dresser_0073.off --out_dir /home/train/neurocross --n_samples 10000
--- Real-time Training Output ---
[2026-06-05 21:58:52] input params: 
[2026-06-05 21:58:52] Namespace(out_dir='/home/train/neurocross', model_name='model', seed=3627473, data_path='/root/.cache/kagglehub/datasets/balraj98/modelnet40-princeton-3d-object-dataset/versions/1/ModelNet40/dresser/train/dresser_0073.off', n_samples=10000, n_points=15000, grid_res=256, nonmnfld_sample_type='gaussian', num_epochs=1, lr=5e-05, grad_clip_norm=10.0, batch_size=1, load_path=None, num_workers=4, persistent_workers=False, fast_nondeterministic=False, log_interval=10, early_stop=False, early_stop_min_steps=1